In [1]:
import requests
import pandas as pd

# Test that we can reach ClinVar
test_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "clinvar",
    "term": "LDLR[gene] AND pathogenic[clinsig]",
    "retmax": 5,
    "retmode": "json"
}

response = requests.get(test_url, params=params)
data = response.json()

ids = data["esearchresult"]["idlist"]
total = data["esearchresult"]["count"]

print(f"Connection to ClinVar: OK")
print(f"Total LDLR pathogenic variants found: {total}")
print(f"First 5 IDs: {ids}")

Connection to ClinVar: OK
Total LDLR pathogenic variants found: 4671
First 5 IDs: ['4813632', '4813630', '4813628', '4813533', '4813277']


In [2]:
def fetch_fh_mutations(gene, max_variants=200):
    """
    Fetch pathogenic FH mutations from ClinVar for a given gene.
    Returns a list of dictionaries with variant info.
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    
    # Step 1: Search for variant IDs
    search_resp = requests.get(
        f"{base_url}esearch.fcgi",
        params={
            "db": "clinvar",
            "term": f"{gene}[gene] AND (pathogenic[clinsig] OR likely pathogenic[clinsig])",
            "retmax": max_variants,
            "retmode": "json"
        }
    )
    ids = search_resp.json()["esearchresult"]["idlist"]
    print(f"{gene}: found {len(ids)} variant IDs")
    
    # Step 2: Fetch summaries for those IDs
    summary_resp = requests.get(
        f"{base_url}esummary.fcgi",
        params={
            "db": "clinvar",
            "id": ",".join(ids),
            "retmode": "json"
        }
    )
    results = summary_resp.json().get("result", {})
    
    # Step 3: Extract the fields we need
    records = []
    for uid in ids:
        entry = results.get(uid, {})
        if not entry:
            continue
            
        # Get clinical significance
        clin_sig = ""
        if "clinical_significance" in entry:
            clin_sig = entry["clinical_significance"].get("description", "")
        
        # Get HGVS name
        title = entry.get("title", "")
        
        # Get variant type
        var_type = entry.get("obj_type", "")
        
        records.append({
            "clinvar_id":  uid,
            "gene":        gene,
            "title":       title,
            "clin_sig":    clin_sig,
            "var_type":    var_type,
        })
    
    return records

# Fetch mutations for all 3 FH genes
all_records = []
for gene in ["LDLR", "APOB", "PCSK9"]:
    records = fetch_fh_mutations(gene, max_variants=200)
    all_records.extend(records)

# Build a DataFrame
df = pd.DataFrame(all_records)
print(f"\nTotal variants collected: {len(df)}")
print(f"\nBreakdown by gene:")
print(df["gene"].value_counts())
print(f"\nBreakdown by significance:")
print(df["clin_sig"].value_counts())

LDLR: found 200 variant IDs
APOB: found 200 variant IDs
PCSK9: found 200 variant IDs

Total variants collected: 600

Breakdown by gene:
gene
LDLR     200
APOB     200
PCSK9    200
Name: count, dtype: int64

Breakdown by significance:
clin_sig
    600
Name: count, dtype: int64


In [3]:
# Let's look at one raw entry to see the actual field names
sample_id = ids[0]
inspect_resp = requests.get(
    "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esummary.fcgi",
    params={
        "db": "clinvar",
        "id": sample_id,
        "retmode": "json"
    }
)
sample = inspect_resp.json()["result"][sample_id]

# Print all available fields
print("Available fields in ClinVar entry:")
for key, value in sample.items():
    print(f"  {key}: {value}")

Available fields in ClinVar entry:
  uid: 4813632
  obj_type: single nucleotide variant
  accession: VCV004813632
  accession_version: VCV004813632.1
  title: NM_000527.5(LDLR):c.1371C>G (p.Asp457Glu)
  variation_set: [{'measure_id': '4924796', 'variation_xrefs': [], 'variation_name': 'NM_000527.5(LDLR):c.1371C>G (p.Asp457Glu)', 'cdna_change': 'c.1371C>G', 'aliases': [], 'variation_loc': [{'status': 'current', 'assembly_name': 'GRCh38', 'chr': '19', 'band': '', 'start': '11113547', 'stop': '11113547', 'inner_start': '', 'inner_stop': '', 'outer_start': '', 'outer_stop': '', 'display_start': '11113547', 'display_stop': '11113547', 'assembly_acc_ver': 'GCF_000001405.38', 'annotation_release': '', 'alt': '', 'ref': ''}, {'status': 'previous', 'assembly_name': 'GRCh37', 'chr': '19', 'band': '', 'start': '11224223', 'stop': '11224223', 'inner_start': '', 'inner_stop': '', 'outer_start': '', 'outer_stop': '', 'display_start': '11224223', 'display_stop': '11224223', 'assembly_acc_ver': 'GCF_0

In [4]:
def fetch_fh_mutations_v2(gene, max_variants=200):
    """
    Improved version using correct ClinVar field names.
    """
    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
    
    # Step 1: Search for variant IDs
    search_resp = requests.get(
        f"{base_url}esearch.fcgi",
        params={
            "db": "clinvar",
            "term": f"{gene}[gene] AND (pathogenic[clinsig] OR likely pathogenic[clinsig])",
            "retmax": max_variants,
            "retmode": "json"
        }
    )
    ids = search_resp.json()["esearchresult"]["idlist"]
    print(f"{gene}: fetching {len(ids)} variants...")
    
    # Step 2: Fetch summaries
    summary_resp = requests.get(
        f"{base_url}esummary.fcgi",
        params={
            "db": "clinvar",
            "id": ",".join(ids),
            "retmode": "json"
        }
    )
    results = summary_resp.json().get("result", {})
    
    # Step 3: Extract fields using correct structure
    records = []
    for uid in ids:
        entry = results.get(uid, {})
        if not entry:
            continue
        
        # Get clinical significance from correct field
        clin_sig = entry.get("germline_classification", {}).get("description", "")
        
        # Get genomic coordinates (GRCh38 only)
        chrom, pos, ref, alt = "", "", "", ""
        variation_set = entry.get("variation_set", [])
        if variation_set:
            for loc in variation_set[0].get("variation_loc", []):
                if loc.get("assembly_name") == "GRCh38":
                    chrom = loc.get("chr", "")
                    pos   = loc.get("start", "")
                    ref   = loc.get("ref", "")
                    alt   = loc.get("alt", "")
                    break
        
        # Get HGVS and other fields
        cdna_change = ""
        if variation_set:
            cdna_change = variation_set[0].get("cdna_change", "")
        
        # Get canonical SPDI (most reliable for sequence retrieval)
        spdi = ""
        if variation_set:
            spdi = variation_set[0].get("canonical_spdi", "")
        
        records.append({
            "clinvar_id":    uid,
            "gene":          gene,
            "title":         entry.get("title", ""),
            "cdna_change":   cdna_change,
            "protein_change": entry.get("protein_change", ""),
            "clin_sig":      clin_sig,
            "var_type":      entry.get("obj_type", ""),
            "chrom":         chrom,
            "pos":           pos,
            "ref":           ref,
            "alt":           alt,
            "spdi":          spdi,
        })
    
    return records

# Re-fetch all three genes with the fixed function
all_records = []
for gene in ["LDLR", "APOB", "PCSK9"]:
    records = fetch_fh_mutations_v2(gene, max_variants=200)
    all_records.extend(records)

# Rebuild DataFrame
df = pd.DataFrame(all_records)

print(f"\nTotal variants collected: {len(df)}")
print(f"\nBreakdown by significance:")
print(df["clin_sig"].value_counts())
print(f"\nBreakdown by variant type:")
print(df["var_type"].value_counts())
print(f"\nSample of first 3 rows:")
print(df[["gene","cdna_change","protein_change","clin_sig","chrom","pos"]].head(3))

LDLR: fetching 200 variants...
APOB: fetching 200 variants...
PCSK9: fetching 200 variants...

Total variants collected: 600

Breakdown by significance:
clin_sig
Uncertain significance                          304
Likely benign                                   193
Pathogenic                                       54
Likely pathogenic                                37
Benign                                            6
Conflicting classifications of pathogenicity      4
                                                  1
Benign/Likely benign                              1
Name: count, dtype: int64

Breakdown by variant type:
var_type
single nucleotide variant    503
Deletion                      50
Duplication                   29
Indel                         12
Insertion                      2
Microsatellite                 2
copy number loss               1
copy number gain               1
Name: count, dtype: int64

Sample of first 3 rows:
   gene     cdna_change              protein

In [5]:
# Step 1: Filter to Pathogenic and Likely Pathogenic only
df_filtered = df[df["clin_sig"].isin(["Pathogenic", "Likely pathogenic"])].copy()

print(f"After P/LP filter: {len(df_filtered)} variants")
print(f"\nBreakdown by gene:")
print(df_filtered["gene"].value_counts())

# Step 2: Keep only single nucleotide variants
# (deletions/insertions are not suitable for simple PCR-RFLP)
df_snv = df_filtered[df_filtered["var_type"] == "single nucleotide variant"].copy()

print(f"\nAfter SNV filter: {len(df_snv)} variants")
print(f"\nBreakdown by gene:")
print(df_snv["gene"].value_counts())

# Step 3: Drop rows missing coordinates or cdna_change
df_clean = df_snv[
    (df_snv["chrom"] != "") &
    (df_snv["pos"]   != "") &
    (df_snv["cdna_change"] != "")
].copy()

# Reset index
df_clean = df_clean.reset_index(drop=True)

print(f"\nAfter removing incomplete entries: {len(df_clean)} variants")
print(f"\nFinal clean dataset sample:")
print(df_clean[["gene","cdna_change","protein_change","chrom","pos"]].head(10))

After P/LP filter: 91 variants

Breakdown by gene:
gene
LDLR     80
APOB      9
PCSK9     2
Name: count, dtype: int64

After SNV filter: 27 variants

Breakdown by gene:
gene
LDLR     21
APOB      5
PCSK9     1
Name: count, dtype: int64

After removing incomplete entries: 27 variants

Final clean dataset sample:
   gene cdna_change              protein_change chrom       pos
0  LDLR    c.408C>G                 D136E, D95E    19  11105314
1  LDLR    c.155G>T                 C138F, C52F    19  11100310
2  LDLR    c.457T>C                F112L, F153L    19  11105363
3  LDLR   c.1840T>G  F446V, F487V, F573V, F614V    19  11116993
4  LDLR   c.1054T>G  C184G, C225G, C311G, C352G    19  11110765
5  LDLR   c.1961T>G  L486R, L527R, L613R, L654R    19  11120207
6  LDLR   c.1865A>T  D454V, D495V, D581V, D622V    19  11120111
7  LDLR   c.1724T>A  L407H, L448H, L534H, L575H    19  11116877
8  LDLR   c.1074T>G  C190W, C231W, C317W, C358W    19  11111527
9  LDLR    c.986G>C  C161S, C202S, C288S, C329S

In [6]:
import os

# Create an output folder for all our results
os.makedirs("results", exist_ok=True)

# Save the clean dataset
df_clean.to_csv("results/fh_mutations_clean.csv", index=False)

# Save the full unfiltered dataset too (useful for reference)
df.to_csv("results/fh_mutations_all.csv", index=False)

print("Files saved:")
print("  results/fh_mutations_clean.csv  — 27 P/LP SNVs ready for RFLP analysis")
print("  results/fh_mutations_all.csv    — full 600 variant dataset")
print()
print("Summary of your working dataset:")
print(f"  Total variants:     {len(df_clean)}")
print(f"  LDLR variants:      {len(df_clean[df_clean['gene']=='LDLR'])}")
print(f"  APOB variants:      {len(df_clean[df_clean['gene']=='APOB'])}")
print(f"  PCSK9 variants:     {len(df_clean[df_clean['gene']=='PCSK9'])}")
print()
print("Phase 2 complete!")


Files saved:
  results/fh_mutations_clean.csv  — 27 P/LP SNVs ready for RFLP analysis
  results/fh_mutations_all.csv    — full 600 variant dataset

Summary of your working dataset:
  Total variants:     27
  LDLR variants:      21
  APOB variants:      5
  PCSK9 variants:     1

Phase 2 complete!
